# YAML - JavaScript

All 6 JavaScript examples from [docs/core/yaml.md](https://platob.github.io/yggdryl/core/yaml/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install @yggdryl/node
```

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('@yggdryl/node')

const value = yaml.loads('symbol: AAPL\nquantity: 2\n')
assert.deepEqual(value, { symbol: 'AAPL', quantity: 2 })

const encoded = yaml.dumps(value)
assert.deepEqual(yaml.loads(encoded), value)

// One document means one.
assert.throws(() => yaml.loads('id: 1\n---\nid: 2\n'), /one YAML document/)

## Documents

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('@yggdryl/node')

const documents = yaml.loadsAll('id: 1\n---\nid: 2\n---\nnull\n')
assert.deepEqual(documents, [{ id: 1 }, { id: 2 }, null])

const encoded = yaml.dumpAll(documents)
assert.match(encoded.toString(), /\n---\n/)
assert.deepEqual(yaml.loadsAll(encoded), documents)

In [ ]:
const assert = require('node:assert/strict')
const { Readable } = require('node:stream')
const { yaml } = require('@yggdryl/node')

async function main() {
  const source = Readable.from([
    Buffer.from('id: 1\n---\n'),
    Buffer.from('items: [1, 2\n'),
  ])
  const documents = []

  // The second document is malformed, and the loop ends after saying so.
  await assert.rejects(async () => {
    for await (const document of yaml.loadAllStream(source)) documents.push(document)
  }, /cumulative byte/)

  assert.deepEqual(documents, [{ id: 1 }])
}

main()

## Tags are read, never written

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('@yggdryl/node')

const encoded = yaml.dumps({ payload: Buffer.from([0, 255]) })
assert.ok(!encoded.toString().includes('!yggdryl'))
assert.match(encoded.toString(), /"\$yggdryl": "bytes"/)
assert.deepEqual(yaml.loads(encoded), { payload: Buffer.from([0, 255]) })

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('@yggdryl/node')

// A machine tag on input is semantic.
assert.deepEqual(yaml.loads('!yggdryl/bytes AP8=\n'), Buffer.from([0, 255]))

// An application tag is an annotation, so the node under it is the value.
assert.deepEqual(yaml.loads('!vendor:quantity {value: 4}\n'), { value: 4 })

// A comment is not read either.
assert.deepEqual(
  yaml.loads('# vendor:attacker\n!vendor:quantity {value: 4}\n'),
  { value: 4 },
)

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('@yggdryl/node')

const collision = { $yggdryl: 'bytes', value: 'AP8=' }

const encoded = yaml.dumps(collision)
assert.match(encoded.toString(), /"\$yggdryl": "mapping"/)
assert.deepEqual(yaml.loads(encoded), collision)